# 🧰 TEMPLATE KOMPETISI: EDA → Preprocessing → Submission
**Cara pakai di hari-H:**
1. Isi bagian **⚙️ KONFIGURASI** (path file, `TARGET`, `ID_COL`, `JENIS`).
2. Jalankan sel dari atas ke bawah (`Shift + Enter`), **baca output tiap sel**.
3. **Submit secepatnya** versi apa adanya (jaring pengaman), **baru** perbaiki di bagian ⭐ UPGRADE.

Nomor bagian = nomor **Resep 12 Langkah** di web Dojo & contekan.
Default-nya sudah diisi untuk data latihan **Titanic**, jadi bisa langsung *Run All* untuk mencoba.

## 1 · Import library

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)   # tampilkan semua kolom

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             mean_squared_error, mean_absolute_error, r2_score)

## 2 · Cari & load data
Di **Kaggle**, sel pertama di bawah menampilkan semua file data yang tersedia beserta path lengkapnya. Copy path-nya ke KONFIGURASI.

In [ ]:
# Daftar semua file data di Kaggle (di laptop sel ini tidak menampilkan apa-apa, itu normal)
for folder, _, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(folder, f))

In [ ]:
# ⚙️ ===== KONFIGURASI: GANTI SESUAI KOMPETISI =====
TRAIN_PATH = 'data/titanic/train.csv'        # Kaggle: '/kaggle/input/NAMA-KOMPETISI/train.csv'
TEST_PATH  = 'data/titanic/test.csv'
SUB_PATH   = 'data/titanic/sample_submission.csv'

TARGET = 'Survived'        # kolom yang ditebak (lihat kolom ke-2 sample_submission)
ID_COL = 'PassengerId'     # kolom ID (kolom ke-1 sample_submission)
JENIS  = 'klasifikasi'     # 'klasifikasi' (tebak kelas) atau 'regresi' (tebak angka)

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SUB_PATH)
print('train:', train.shape, '| test:', test.shape, '| sample:', sample.shape)
sample.head()      # <- format file yang harus dikirim

## 3 · Kenalan dengan data

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.describe(exclude='number')

## 4 · Missing value (train & test)

In [ ]:
kosong = pd.DataFrame({
    'train_jumlah': train.isnull().sum(),
    'train_persen': (train.isnull().mean() * 100).round(1),
    'test_jumlah': test.isnull().sum(),
})
kosong[(kosong['train_jumlah'] > 0) | (kosong['test_jumlah'] > 0)].sort_values('train_persen', ascending=False)

## 5 · Duplikat

In [ ]:
print('Baris duplikat di train:', train.duplicated().sum())

## 6 · Target

In [ ]:
if JENIS == 'klasifikasi':
    print(train[TARGET].value_counts())
    print(train[TARGET].value_counts(normalize=True).round(3))
    print('Baseline (tebak kelas terbanyak):', round(train[TARGET].value_counts(normalize=True).max(), 4))
    sns.countplot(data=train, x=TARGET)
else:
    print(train[TARGET].describe())
    print('Skewness:', round(train[TARGET].skew(), 2), '(> 1 = miring kanan → coba log target)')
    sns.histplot(data=train, x=TARGET, kde=True)
plt.title('Distribusi target')
plt.show()

## 7 · Kolom angka: sebaran & nilai aneh

In [ ]:
num_cols = train.select_dtypes(include='number').columns.drop([TARGET, ID_COL], errors='ignore').tolist()
cat_cols = train.select_dtypes(exclude='number').columns.drop([TARGET, ID_COL], errors='ignore').tolist()
print('Numerik   :', num_cols)
print('Kategorikal:', cat_cols)

In [ ]:
train[num_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

## 8 · Kolom teks: jumlah nilai unik & isinya

In [ ]:
# Ratusan nilai unik = teks bebas / ID / tanggal. Nilai mirip-mirip (jakarta vs Jakarta) = perlu dirapikan
train[cat_cols].nunique().sort_values(ascending=False)

In [ ]:
for col in cat_cols:
    if train[col].nunique() <= 30:
        print(f'--- {col} ---')
        print(train[col].value_counts(dropna=False).head(15), '\n')

## 9 · Hubungan fitur ↔ target

In [ ]:
for col in cat_cols:
    if train[col].nunique() <= 15:
        if JENIS == 'klasifikasi':
            print(pd.crosstab(train[col], train[TARGET], normalize='index').round(2), '\n')
        else:
            print(train.groupby(col)[TARGET].agg(['mean', 'median', 'count']).round(1), '\n')

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(train.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Korelasi')
plt.show()

### ✍️ Kesimpulan EDA (isi 3–6 poin)
-

## 10 · Preprocessing
### 10a. Bersih-bersih manual (lakukan ke train DAN test)
Aktifkan (hapus `#`) dan sesuaikan baris yang perlu berdasarkan temuan EDA.

In [ ]:
train = train.drop_duplicates()

for df in [train, test]:
    pass   # <- baris ini biarkan saja (supaya loop tidak kosong)
    # df['kota'] = df['kota'].str.strip().str.title()                              # rapikan ejaan
    # df.loc[(df['umur'] <= 0) | (df['umur'] > 100), 'umur'] = np.nan              # nilai mustahil -> NaN
    # df.loc[df['tahun'] == 0, 'tahun'] = np.nan                                   # 0 = tidak diketahui
    # df['tahun_daftar'] = pd.to_datetime(df['tanggal'], errors='coerce').dt.year  # tanggal -> tahun
    # df['kolom_kosong'] = df['kolom'].isnull().astype(int)                        # penanda missing
    # df['FamilySize'] = df['SibSp'] + df['Parch'] + 1                             # fitur baru

### 10b. Tentukan kolom yang dibuang

In [ ]:
cat_all = train.select_dtypes(exclude='number').columns.drop([TARGET, ID_COL], errors='ignore')
saran = [c for c in cat_all if train[c].nunique() > 100 or train[c].nunique() > 0.5 * len(train)]  # teks bebas/ID/tanggal
saran += [c for c in train.columns if train[c].isnull().mean() > 0.6 and c not in saran]           # hampir kosong
saran += [c for c in train.columns if train[c].nunique() <= 1 and c not in saran]                  # isinya cuma 1 nilai
print('Saran kolom dibuang:', saran)

BUANG = [ID_COL] + saran      # <- boleh diedit manual, contoh: [ID_COL, 'Name', 'Ticket', 'Cabin']

### 10c. Siapkan X, y dan pipeline

In [ ]:
X = train.drop(columns=[TARGET] + BUANG, errors='ignore')
y = train[TARGET]
X_test = test.drop(columns=BUANG, errors='ignore')

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
print('Fitur angka:', num_cols)
print('Fitur teks :', cat_cols)

preprocess = ColumnTransformer([
    ('num', Pipeline([('isi_kosong', SimpleImputer(strategy='median')),
                      ('skala', StandardScaler())]), num_cols),
    ('kat', Pipeline([('isi_kosong', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
], sparse_threshold=0)   # sparse_threshold=0 -> hasilnya tabel biasa, aman untuk semua model

if JENIS == 'klasifikasi':
    model_dasar = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
else:
    model_dasar = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

model = Pipeline([('prep', preprocess), ('model', model_dasar)])

## 11 · Validasi

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if JENIS == 'klasifikasi' else None)

model.fit(X_tr, y_tr)
pred_val = model.predict(X_val)

if JENIS == 'klasifikasi':
    print('Accuracy :', round(accuracy_score(y_val, pred_val), 4))
    print('F1 macro :', round(f1_score(y_val, pred_val, average='macro'), 4))
    print('Baseline :', round(y_val.value_counts(normalize=True).max(), 4), '(tebak kelas terbanyak)')
else:
    print('RMSE:', round(np.sqrt(mean_squared_error(y_val, pred_val)), 3))
    print('MAE :', round(mean_absolute_error(y_val, pred_val), 3))
    print('R2  :', round(r2_score(y_val, pred_val), 4))
    print('Baseline RMSE:', round(np.sqrt(mean_squared_error(y_val, np.full(len(y_val), y_tr.mean()))), 3), '(tebak rata-rata)')

## 12 · Model final → prediksi test → submission

In [ ]:
model.fit(X, y)                        # latih ulang pakai SEMUA data train
pred_test = model.predict(X_test)
# Kalau metrik kompetisi ROC-AUC / LogLoss -> kirim PROBABILITAS:
# pred_test = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: pred_test})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.head()

In [ ]:
# Cek format sebelum submit
print('Kolom sama dengan sample?  ', list(submission.columns) == list(sample.columns), '|', list(sample.columns))
print('Jumlah baris sama?         ', len(submission) == len(sample))
print('Ada prediksi kosong?       ', submission[TARGET].isnull().any())
print('ID sama dengan sample?     ', set(submission[ID_COL]) == set(sample[ID_COL]))

## ⭐ UPGRADE (kalau masih ada waktu, satu per satu, cek skor validasi tiap kali)
Setelah mengubah sesuatu, jalankan ulang **10c → 11**. Kalau skor validasi naik, jalankan **12** lalu submit.

In [ ]:
# A. Ganti model (pilih satu, lalu jalankan ulang bagian 11)
# model = Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=1000))])            # klasifikasi
# model = Pipeline([('prep', preprocess), ('model', LinearRegression())])                          # regresi
# model = Pipeline([('prep', preprocess), ('model', RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1))])

In [ ]:
# B. Regresi dengan target miring kanan -> latih di log(target)
# model = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)

In [ ]:
# C. Skor yang lebih bisa dipercaya: cross-validation 5 lipatan
# scoring: 'accuracy', 'f1_macro', 'roc_auc' (klasifikasi) / 'neg_root_mean_squared_error' (regresi)
# skor_cv = cross_val_score(model, X, y, cv=5, scoring='accuracy')
# print(skor_cv.round(4), '→ rata-rata:', skor_cv.mean().round(4))